In [ ]:
# 运行此代码单元格来挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# 成功后 Google Drive内容将可以在以下路径访问：
# '/content/drive/My Drive/'
# 假如文件夹名为 'my_project_data' 并且存储在Google Drive的根目录下
# 在Colab中就可以通过 '/content/drive/My Drive/my_project_data/' 来访问它。
drive_path = '/content/drive/My Drive/自选题' # 定义想要切换到的文件夹路径

if os.path.exists(drive_path):
    os.chdir(drive_path)
    print(f"当前工作目录已切换至: {os.getcwd()}")
else:
    print(f"错误: 路径不存在 {drive_path}")

Mounted at /content/drive
当前工作目录已切换至: /content/drive/My Drive/自选题


# Experiment: EMCAD Improvement On ETIS

这一册基于真实 ETIS 数据，只围绕 prediction head 的轻量结构改进做对比，而不重复完整 baseline。


## 加载基础工具

这里只保留环境与结果保存工具，真实改进实验逻辑都写在本 notebook 内。


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "torch_installed": true,
  "cuda_available": true
}


## 读取共享实验配置

这里直接读取 `00` 已保存的统一配置，继续沿用 `01` 的 ETIS 数据路径和统一训练参数。


In [ ]:
PROJECT_CONFIG = load_project_config()
PROJECT_CONFIG


{'dataset': 'ETIS',
 'task': 'Polyp Segmentation',
 'paper_repo': 'https://github.com/SLDGroup/EMCAD',
 'baseline_repos': {'Swin-Unet': 'https://github.com/HuCaoFighting/Swin-Unet',
  'U-Net': 'https://github.com/milesial/Pytorch-UNet'},
 'emcad_scale': 'PVT-EMCAD-B0',
 'metrics': ['Dice'],
 'fixed_visual_sample': '100.png',
 'etis_paths': {'root': '/content/drive/My Drive/自选题/data/ETIS',
  'train_images': '/content/drive/My Drive/自选题/data/ETIS/train/images',
  'train_masks': '/content/drive/My Drive/自选题/data/ETIS/train/masks',
  'val_images': '/content/drive/My Drive/自选题/data/ETIS/val/images',
  'val_masks': '/content/drive/My Drive/自选题/data/ETIS/val/masks',
  'test_images': '/content/drive/My Drive/自选题/data/ETIS/test/images',
  'test_masks': '/content/drive/My Drive/自选题/data/ETIS/test/masks',
  'train_list': '/content/drive/My Drive/自选题/data/ETIS/train_list_etis.txt',
  'val_list': '/content/drive/My Drive/自选题/data/ETIS/val_list_etis.txt',
  'test_list': '/content/drive/My Drive/自选题/

## 读取 EMCAD baseline 结果

baseline 结果固定来自 `01_emcad_full_training.ipynb`。


In [ ]:
baseline_result_path = ARTIFACTS / "records" / "emcad_etis_results.json"
baseline_reference = json.loads(baseline_result_path.read_text(encoding="utf-8")) if baseline_result_path.exists() else {}
baseline_reference.get("test_summary", {})


{'loss': 0.6782, 'dice': 0.8787}

## 复用 baseline 数据与 EMCAD 组件

这里继续复用 `01` 的 ETIS 数据流与 EMCAD 组件，只单独定义改进过的 prediction head。


In [ ]:
import json
from pathlib import Path

baseline_nb = json.loads(Path("01_emcad_full_training.ipynb").read_text(encoding="utf-8"))
for idx in [3, 5, 7, 9]:
    exec("".join(baseline_nb["cells"][idx]["source"]), globals())


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "torch_installed": true,
  "cuda_available": true
}


## 定义改进模块

这里不重写整个 baseline，只定义 prediction head 的差异：

- baseline：单头预测
- improvement：可学习融合头


In [ ]:
class LearnableFusionHead(nn.Module):
    def __init__(self, in_channels=32, branches=3):
        super().__init__()
        self.projections = nn.ModuleList([nn.Conv2d(in_channels, 1, kernel_size=1) for _ in range(branches)])
        self.weights = nn.Parameter(torch.ones(branches))

    def forward(self, x):
        branch_logits = [layer(x) for layer in self.projections]
        fusion_weights = torch.softmax(self.weights, dim=0)
        return sum(weight * logit for weight, logit in zip(fusion_weights, branch_logits))

class ImprovedEMCADModel(nn.Module):
    def __init__(self, kernel_sizes=(1, 3, 5)):
        super().__init__()
        self.encoder = PVTv2B0Encoder()
        self.dec3 = EMCADDecoderBlock(256, 160, 160, kernel_sizes=kernel_sizes)
        self.dec2 = EMCADDecoderBlock(160, 64, 64, kernel_sizes=kernel_sizes)
        self.dec1 = EMCADDecoderBlock(64, 32, 32, kernel_sizes=kernel_sizes)
        self.head = LearnableFusionHead(in_channels=32, branches=3)

    def load_pretrained_backbone(self, weight_path):
        self.encoder.load_pretrained(weight_path)

    def forward(self, x):
        skip1, skip2, skip3, bottleneck = self.encoder(x)
        x = self.dec3(bottleneck, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)
        logits = self.head(x)
        return F.interpolate(logits, scale_factor=4, mode="bilinear", align_corners=False)


## 说明 baseline 与改进版的结构差异

这一段明确改进前后变化的范围和预期收益。


In [ ]:
improvement_design = {
    "baseline_source": "01_emcad_full_training.ipynb",
    "baseline_summary": "EMCAD baseline uses a single 1x1 prediction head after decoder refinement.",
    "improvement_change": "Replace the single prediction head with a learnable multi-branch fusion head.",
    "expected_benefit": "Improve final decoder output aggregation without changing the whole backbone and decoder stack.",
}
improvement_design


{'baseline_source': '01_emcad_full_training.ipynb',
 'baseline_summary': 'EMCAD baseline uses a single 1x1 prediction head after decoder refinement.',
 'improvement_change': 'Replace the single prediction head with a learnable multi-branch fusion head.',
 'expected_benefit': 'Improve final decoder output aggregation without changing the whole backbone and decoder stack.'}

## 在 ETIS 上训练改进版 EMCAD

这里继续使用同一套 ETIS 训练、验证和测试流程，对改进版完成完整训练与评估。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PROJECT_CONFIG["train"]
train_loader, val_loader, test_loader = build_dataloaders(cfg)
weight_path = Path(PROJECT_CONFIG["pvt_pretrained_path"])
assert weight_path.exists(), f"未找到预训练权重: {weight_path}"

improved_model = ImprovedEMCADModel().to(device)
improved_model.load_pretrained_backbone(weight_path)
improvement_history, improvement_best_val_dice = train_model(improved_model, train_loader, val_loader, cfg, device=device)
improvement_val_summary, improvement_val_rows = evaluate_loader(improved_model, val_loader, device=device)
improvement_test_summary, improvement_test_rows = evaluate_loader(improved_model, test_loader, device=device)
improvement_test_summary


{'loaded_weight_path': '/content/drive/My Drive/自选题/data/pvt_pretrained_pth/pvt_v2_b0.pth', 'missing_keys': 0, 'unexpected_keys': 2}
{'epoch': 1, 'train_loss': 1.615, 'train_dice': 0.1179, 'val_loss': 1.629, 'val_dice': 0.0852}
{'epoch': 2, 'train_loss': 1.5397, 'train_dice': 0.2038, 'val_loss': 1.5928, 'val_dice': 0.1567}
{'epoch': 3, 'train_loss': 1.4937, 'train_dice': 0.2772, 'val_loss': 1.4963, 'val_dice': 0.2816}
{'epoch': 4, 'train_loss': 1.4619, 'train_dice': 0.3794, 'val_loss': 1.456, 'val_dice': 0.3935}
{'epoch': 5, 'train_loss': 1.4363, 'train_dice': 0.441, 'val_loss': 1.4228, 'val_dice': 0.493}
{'epoch': 6, 'train_loss': 1.4084, 'train_dice': 0.5392, 'val_loss': 1.4065, 'val_dice': 0.4617}
{'epoch': 7, 'train_loss': 1.3929, 'train_dice': 0.5858, 'val_loss': 1.405, 'val_dice': 0.5331}
{'epoch': 8, 'train_loss': 1.3727, 'train_dice': 0.6287, 'val_loss': 1.374, 'val_dice': 0.5045}
{'epoch': 9, 'train_loss': 1.362, 'train_dice': 0.6545, 'val_loss': 1.3756, 'val_dice': 0.5769}
{'

{'loss': 0.8946, 'dice': 0.8473}

## 导出改进版的统一测试样本结果

这里固定导出与 baseline 相同的测试样本，便于直接比较改进前后的预测差异。


In [9]:
improvement_history_path = ARTIFACTS / "figures" / "emcad_improved_history.png"
saved_improvement_history = save_history_figure(improvement_history, improvement_history_path)
improvement_visual_path = ARTIFACTS / "figures" / "emcad_improved_visual_sample.png"
saved_improvement_visual = export_visualization(
    improved_model,
    default_visual_sample(),
    cfg["image_size"],
    improvement_visual_path,
    device=device,
)
saved_improvement_visual


'/content/drive/My Drive/自选题/artifacts/figures/emcad_improved_visual_sample.png'

## 生成 baseline 与改进版对比

这里把 `01` 的 baseline 结果与当前改进版结果放到一起，突出改动点和结果解释。


In [10]:
comparison = []
if baseline_reference:
    comparison.append(
        {
            "variant": "EMCAD baseline",
            "best_val_dice": baseline_reference["best_val_dice"],
            "test_summary": baseline_reference["test_summary"],
            "difference": "Reference baseline result",
            "visual_path": baseline_reference["visual_path"],
        }
    )

comparison.append(
    {
        "variant": "Improved learnable fusion head",
        "best_val_dice": improvement_best_val_dice,
        "test_summary": improvement_test_summary,
        "difference": "Replace the single prediction head with a multi-branch learnable fusion head",
        "visual_path": saved_improvement_visual,
    }
)
comparison


[{'variant': 'EMCAD baseline',
  'best_val_dice': 0.8128,
  'test_summary': {'loss': 0.6782, 'dice': 0.8787},
  'difference': 'Reference baseline result',
  'visual_path': '/content/drive/My Drive/自选题/artifacts/figures/emcad_visual_sample.png'},
 {'variant': 'Improved learnable fusion head',
  'best_val_dice': 0.8003,
  'test_summary': {'loss': 0.8946, 'dice': 0.8473},
  'difference': 'Replace the single prediction head with a multi-branch learnable fusion head',
  'visual_path': '/content/drive/My Drive/自选题/artifacts/figures/emcad_improved_visual_sample.png'}]

## 保存改进实验记录

这里会把改进设计、统一测试样本和对比结果一起写入 records。


In [11]:
improvement_checkpoint_path = ARTIFACTS / "checkpoints" / "emcad_improved_etis_best.pth"
torch.save(improved_model.state_dict(), improvement_checkpoint_path)

save_json(
    {
        "dataset": "ETIS",
        "pretrained_path": str(weight_path),
        "improvement_design": improvement_design,
        "history": improvement_history,
        "best_val_dice": improvement_best_val_dice,
        "val_summary": improvement_val_summary,
        "test_summary": improvement_test_summary,
        "val_rows": improvement_val_rows,
        "test_rows": improvement_test_rows,
        "history_figure_path": saved_improvement_history,
        "visual_sample": default_visual_sample(),
        "visual_path": saved_improvement_visual,
        "checkpoint_path": str(improvement_checkpoint_path),
        "comparison": comparison,
    },
    ARTIFACTS / "records" / "improvement_experiment_etis.json",
)
